In [ ]:
!pip install -q \
    fastapi uvicorn[standard] \
    transformers accelerate \
    torch torchvision \
    Pillow \
    qwen-vl-utils \
    nest_asyncio

In [ ]:
import io
import json
import base64
import uuid
from pathlib import Path
 
import torch
import numpy as np
from PIL import Image, ImageOps
 
import nest_asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, HTTPException
from fastapi.responses import JSONResponse
 
nest_asyncio.apply()

In [ ]:
# 1. Install zrok directly into the Kaggle environment
!curl -sSf https://get.openziti.io/install.bash | sudo bash -s zrok

In [ ]:
# 2. Retrieve your secret and enable zrok
from kaggle_secrets import UserSecretsClient

ZROK_TOKEN = UserSecretsClient().get_secret("ZROK_TOKEN")
!zrok enable {ZROK_TOKEN}

In [ ]:
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
CROPS_DIR       = Path("/kaggle/working/crops")
CROPS_DIR.mkdir(parents=True, exist_ok=True)
 
RTDETR_MODEL_ID  = "PekingU/rtdetr_r50vd"
SIGLIP_MODEL_ID  = "google/siglip2-so400m-patch16-naflex"
QWEN_MODEL_ID    = "Qwen/Qwen3-VL-4B-Instruct"   # swap to 2B for runtime session
 
# Map model labels into a stable furniture-centric label set.
FURNITURE_LABEL_ALIASES = {
    "chair": "chair",
    "sofa": "sofa",
    "couch": "sofa",
    "bed": "bed",
    "dining_table": "dining_table",
    "dining table": "dining_table",
    "potted_plant": "potted_plant",
    "potted plant": "potted_plant",
    "tv": "tv",
    "tvmonitor": "tv",
    "toilet": "toilet",
    "clock": "clock",
}

def normalize_label_name(name: str) -> str:
    return name.lower().strip().replace("-", "_").replace(" ", "_")


def build_furniture_label_lookup(id2label: dict) -> dict[int, str]:
    lookup = {}
    for raw_id, raw_label in id2label.items():
        canonical = FURNITURE_LABEL_ALIASES.get(normalize_label_name(str(raw_label)))
        if canonical:
            lookup[int(raw_id)] = canonical

    if not lookup:
        raise RuntimeError("RT-DETR did not expose any expected furniture labels.")

    return lookup
 
MASTER_TAXONOMY = {
    "style": [
        "new_classic", "modern", "scandinavian", "industrial",
        "bohemian", "minimalist", "japandi", "coastal",
        "rustic", "art_deco", "traditional", "heritage_fusion", "desert_modern",
    ],
    "material": [
        "solid_wood", "engineered_wood", "metal", "glass",
        "leather", "faux_leather", "fabric", "velvet",
        "linen", "rattan", "marble", "travertine", "plastic",
    ],
    "color": [
        "black", "white", "gray", "brown", "beige",
        "blue", "navy", "green", "red", "orange",
        "yellow", "pink", "purple", "gold",
    ],
    "room_fit": [
        "salon", "living_room", "bedroom", "dining_room",
        "office", "outdoor", "kids_room", "bathroom",
    ],
    "complementary_styles": [
        "modern", "scandinavian", "industrial", "bohemian",
        "minimalist", "japandi", "coastal", "rustic",
        "art_deco", "heritage_fusion", "desert_modern",
    ],
}
 
QWEN_SYSTEM_PROMPT = f"""
You are a professional interior design cataloger for an Egyptian furniture marketplace.
Analyze the furniture item in the image and return ONLY a valid JSON object.
No explanation, no markdown fences, no extra text — raw JSON only.
 
Schema:
{{
  "color":                <string — one of: {MASTER_TAXONOMY['color']}>,
  "material":             <string — one of: {MASTER_TAXONOMY['material']}>,
  "style":                <string — one of: {MASTER_TAXONOMY['style']}>,
  "room_fit":             <list   — values from: {MASTER_TAXONOMY['room_fit']}>,
  "complementary_styles": <list   — values from: {MASTER_TAXONOMY['complementary_styles']}>,
  "description":          <string — one sentence in English>
}}
 
Rules:
- Never invent values outside the allowed lists.
- Pick the single closest allowed value when uncertain.
- room_fit and complementary_styles must always be JSON arrays.
- Use underscores exactly as written in the allowed values.
- Return raw JSON only.
""".strip()
 
print(f"Device : {DEVICE}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
from transformers import RTDetrImageProcessor, RTDetrForObjectDetection
import torch

RTDETR_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Loading RT-DETR...")
rtdetr_processor = RTDetrImageProcessor.from_pretrained(RTDETR_MODEL_ID)
rtdetr_model     = RTDetrForObjectDetection.from_pretrained(
    RTDETR_MODEL_ID,
    torch_dtype=RTDETR_DTYPE,
).to(DEVICE)

rtdetr_model.eval()
FURNITURE_LABEL_LOOKUP = build_furniture_label_lookup(rtdetr_model.config.id2label)

print("RT-DETR ready.")
print(f"Resolved furniture labels: {FURNITURE_LABEL_LOOKUP}")

In [ ]:
from transformers import AutoProcessor, AutoModel
 
print("Loading SigLIP2...")
SIGLIP_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
siglip_processor = AutoProcessor.from_pretrained(SIGLIP_MODEL_ID)
siglip_model     = AutoModel.from_pretrained(
    SIGLIP_MODEL_ID, torch_dtype=SIGLIP_DTYPE
).to(DEVICE)
siglip_model.eval()
print("SigLIP2 ready.")

In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor as QwenProcessor
from qwen_vl_utils import process_vision_info

print("Loading Qwen3-VL...")
QWEN_DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,
    torch_dtype=QWEN_DTYPE,
    device_map="auto",
)
qwen_processor = QwenProcessor.from_pretrained(QWEN_MODEL_ID)
print("Qwen3-VL ready.")

In [ ]:
def _pil_from_upload(data: bytes) -> Image.Image:
    image = Image.open(io.BytesIO(data))
    image = ImageOps.exif_transpose(image)
    return image.convert("RGB")
 
 
def _pil_to_b64(image: Image.Image) -> str:
    buf = io.BytesIO()
    image.save(buf, format="JPEG")
    return base64.b64encode(buf.getvalue()).decode()
 
 
def _box_iou(box_a: list[int], box_b: list[int]) -> float:
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    if inter_area == 0:
        return 0.0

    area_a = max(1, (ax2 - ax1) * (ay2 - ay1))
    area_b = max(1, (bx2 - bx1) * (by2 - by1))
    return inter_area / float(area_a + area_b - inter_area)


def _dedupe_crops(crops: list[dict], iou_threshold: float = 0.85) -> list[dict]:
    kept = []
    for crop in sorted(crops, key=lambda item: item["confidence"], reverse=True):
        overlaps_existing = any(
            crop["label"] == existing["label"] and _box_iou(crop["bbox"], existing["bbox"]) >= iou_threshold
            for existing in kept
        )
        if overlaps_existing:
            continue
        kept.append(crop)
    return kept
 
 
def _move_inputs(batch: dict, device: str, float_dtype: torch.dtype | None = None) -> dict:
    moved = {}
    for key, value in batch.items():
        if torch.is_tensor(value):
            if float_dtype is not None and torch.is_floating_point(value):
                moved[key] = value.to(device=device, dtype=float_dtype)
            else:
                moved[key] = value.to(device=device)
        else:
            moved[key] = value
    return moved
 
 
def run_detect(image: Image.Image, threshold: float = 0.35, debug: bool = False) -> list[dict] | dict:
    """RT-DETR: detect furniture-like objects, then return deduped crops as base64."""
    inputs = rtdetr_processor(images=image, return_tensors="pt")
    inputs = _move_inputs(inputs, DEVICE, RTDETR_DTYPE)
 
    with torch.no_grad():
        outputs = rtdetr_model(**inputs)
 
    results = rtdetr_processor.post_process_object_detection(
        outputs,
        target_sizes=torch.tensor([image.size[::-1]]),
        threshold=threshold,
    )[0]
 
    crops = []
    raw_detections = []
    w, h  = image.size
    min_edge = max(40, int(min(w, h) * 0.04))
 
    for score, label_id, box in zip(
        results["scores"], results["labels"], results["boxes"]
    ):
        lid = int(label_id.item())
        raw_label = normalize_label_name(str(rtdetr_model.config.id2label.get(lid, lid)))
        label = FURNITURE_LABEL_LOOKUP.get(lid)
 
        x1, y1, x2, y2 = [int(round(v)) for v in box.tolist()]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)

        raw_detections.append({
            "label_id": lid,
            "raw_label": raw_label,
            "mapped_label": label,
            "confidence": round(float(score.item()), 3),
            "bbox": [x1, y1, x2, y2],
        })

        if label is None:
            continue
 
        if (x2 - x1) < min_edge or (y2 - y1) < min_edge:
            continue
 
        crop    = image.crop((x1, y1, x2, y2))
        crop_id = str(uuid.uuid4())[:12]
 
        crops.append({
            "crop_id":    crop_id,
            "label":      label,
            "confidence": round(float(score.item()), 3),
            "bbox":       [x1, y1, x2, y2],
            "crop_b64":   _pil_to_b64(crop),          # JPEG crop encoded as base64
        })
 
    crops = _dedupe_crops(crops)
    if debug:
        return {
            "crops": crops,
            "count": len(crops),
            "raw_detections": raw_detections,
            "raw_count": len(raw_detections),
            "furniture_label_lookup": FURNITURE_LABEL_LOOKUP,
        }
    return crops
 
 
def _siglip_extract_features(features):
    if hasattr(features, "image_embeds") and features.image_embeds is not None:
        return features.image_embeds
    if hasattr(features, "pooler_output") and features.pooler_output is not None:
        return features.pooler_output
    if hasattr(features, "last_hidden_state") and features.last_hidden_state is not None:
        return features.last_hidden_state
    return features
 
 
def _siglip_to_vector(features: torch.Tensor | np.ndarray | list) -> list[float]:
    features = _siglip_extract_features(features)
    features = torch.as_tensor(features)
    if features.ndim == 1:
        features = features.unsqueeze(0)
    elif features.ndim == 3:
        features = features.mean(dim=1)
    elif features.ndim > 3:
        features = features.reshape(features.shape[0], -1)
 
    vec = features[0].float()
    vec = vec / (torch.linalg.vector_norm(vec) + 1e-12)
    return vec.cpu().tolist()
 
 
def run_embed(image: Image.Image) -> list[float]:
    """SigLIP2: embed image crop, return one normalized image vector."""
    inputs = siglip_processor(images=image, return_tensors="pt")
    inputs = _move_inputs(inputs, DEVICE, SIGLIP_DTYPE)
 
    if hasattr(siglip_model, "get_image_features"):
        with torch.no_grad():
            features = siglip_model.get_image_features(**inputs)
        return _siglip_to_vector(features)
 
    with torch.no_grad():
        outputs = siglip_model(**inputs)
 
    if hasattr(outputs, "image_embeds") and outputs.image_embeds is not None:
        features = outputs.image_embeds
    elif hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
        features = outputs.pooler_output
    elif hasattr(outputs, "last_hidden_state") and outputs.last_hidden_state is not None:
        features = outputs.last_hidden_state
    elif hasattr(siglip_model, "vision_model"):
        with torch.no_grad():
            vision_outputs = siglip_model.vision_model(**inputs)
        if hasattr(vision_outputs, "pooler_output") and vision_outputs.pooler_output is not None:
            features = vision_outputs.pooler_output
        elif hasattr(vision_outputs, "last_hidden_state") and vision_outputs.last_hidden_state is not None:
            features = vision_outputs.last_hidden_state
        else:
            raise RuntimeError("SigLIP2 vision model did not expose image embeddings.")
    else:
        raise RuntimeError("SigLIP2 model did not expose image embeddings.")
 
    return _siglip_to_vector(features)
 
 
def run_tag(image: Image.Image, max_retries: int = 2) -> dict | None:
    """Qwen3-VL: tag image crop, return taxonomy JSON."""
    # Save crop to temp file — Qwen needs a file path
    tmp_path = f"/kaggle/working/tmp_{uuid.uuid4().hex[:8]}.jpg"
    image.save(tmp_path)
 
    messages = [
        {"role": "system", "content": QWEN_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{tmp_path}"},
                {"type": "text",  "text": "Analyze this furniture item and return the JSON."},
            ],
        },
    ]
 
    text_prompt = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = qwen_processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        return_tensors="pt",
    )
    inputs = _move_inputs(inputs, DEVICE, QWEN_DTYPE)
 
    for attempt in range(max_retries + 1):
        try:
            with torch.no_grad():
                output_ids = qwen_model.generate(
                    **inputs,
                    max_new_tokens=256,
                    temperature=0.1,
                    do_sample=True,
                )
 
            trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
            raw     = qwen_processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
 
            # Strip accidental markdown fences
            raw = raw.strip("`").strip()
            if raw.startswith("json"):
                raw = raw[4:].strip()
 
            metadata = json.loads(raw)
 
            required = ["color", "material", "style", "room_fit", "complementary_styles"]
            for field in required:
                assert field in metadata, f"Missing: {field}"
 
            Path(tmp_path).unlink(missing_ok=True)
            return metadata
 
        except Exception as e:
            print(f"  Qwen attempt {attempt + 1} failed: {e}")
 
    Path(tmp_path).unlink(missing_ok=True)
    return None

In [ ]:
app = FastAPI(title="IntelliRoom Kaggle Model Server")
 
 
@app.get("/health")
async def health():
    return {
        "status": "ok",
        "models": {
            "rtdetr":  RTDETR_MODEL_ID,
            "siglip":  SIGLIP_MODEL_ID,
            "qwen":    QWEN_MODEL_ID,
        },
        "device": DEVICE,
        "vram_free_gb": round(
            torch.cuda.memory_reserved(0) / 1e9, 2
        ) if torch.cuda.is_available() else None,
    }
 
 
@app.post("/detect")
async def detect(image: UploadFile, threshold: float = 0.35, debug: bool = False):
    """
    Input : image file
    Output: list of detected furniture crops as base64 JPEGs
    """
    try:
        data  = await image.read()
        pil   = _pil_from_upload(data)
        result = run_detect(pil, threshold, debug=debug)
        if debug:
            return result
        return {"crops": result, "count": len(result)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
 
 
@app.post("/embed")
async def embed(image: UploadFile):
    """
    Input : image file (should be a crop, not a full room photo)
    Output: 768D normalized float vector
    """
    try:
        data   = await image.read()
        pil    = _pil_from_upload(data)
        vector = run_embed(pil)
        return {"embedding": vector, "dim": len(vector)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
 
 
@app.post("/tag")
async def tag(image: UploadFile):
    """
    Input : image file (crop)
    Output: taxonomy JSON {color, material, style, room_fit, complementary_styles, description}
    """
    try:
        data     = await image.read()
        pil      = _pil_from_upload(data)
        metadata = run_tag(pil)
        if metadata is None:
            raise HTTPException(status_code=422, detail="Qwen failed to produce valid JSON after retries.")
        return metadata
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
 
 
@app.post("/embed_b64")
async def embed_b64(payload: dict):
    """
    Alternative embed endpoint that accepts base64 image string.
    Used by orchestrator to embed crops returned from /detect without re-uploading.
    Input : {"image_b64": "<base64 string>"}
    Output: {"embedding": [...], "dim": 768}
    """
    try:
        img_bytes = base64.b64decode(payload["image_b64"])
        pil       = _pil_from_upload(img_bytes)
        vector    = run_embed(pil)
        return {"embedding": vector, "dim": len(vector)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
 
 
@app.post("/tag_b64")
async def tag_b64(payload: dict):
    """
    Alternative tag endpoint that accepts base64 image string.
    Input : {"image_b64": "<base64 string>"}
    Output: taxonomy JSON
    """
    try:
        img_bytes = base64.b64decode(payload["image_b64"])
        pil       = _pil_from_upload(img_bytes)
        metadata  = run_tag(pil)
        if metadata is None:
            raise HTTPException(status_code=422, detail="Qwen failed to produce valid JSON.")
        return metadata
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
# ── CELL 9: Start Server + zrok Tunnel ───────────────────────

import threading
import time

# Step 1: Start uvicorn in a background thread
def start_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=start_uvicorn, daemon=True)
server_thread.start()

# Step 2: Wait until port 8000 is actually listening
import socket

def wait_for_port(port: int, timeout: float = 30.0):
    start = time.time()
    while time.time() - start < timeout:
        try:
            with socket.create_connection(("localhost", port), timeout=1):
                return True
        except OSError:
            time.sleep(0.5)
    raise RuntimeError(f"Server didn't start on port {port} within {timeout}s")

wait_for_port(8000)
print("Server is up on port 8000.")

# Step 3: Now start zrok (blocking — keep this cell running)
!zrok share public http://localhost:8000